# Final held-out test evaluation

Evaluate the XGBoost detector selected in notebook 05 on the saved seed-42 test subset. The results apply to this sampled test set. Capture-level independence was not verified.

## Lock the model before reading the test subset

Reproduce notebook 05 using its training-only balanced sample weights, ordered feature selection, and CPU configuration. The saved model in `data/models/` is the logistic baseline, so fit XGBoost once on the 100,000 training rows. The cutoff is fixed at 0.5, with scores equal to the cutoff classified as attack.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.batch.baseline_data import (
    ATTACK_THRESHOLD, SUBSET_DIR, TRAINING_PATH, binary_targets,
    load_features, select_model_features, validation_overlap,
)
from src.batch.validate import score_predictions

training, training_labels = load_features(TRAINING_PATH)
y_train = binary_targets(training_labels, "Training").to_numpy()
feature_columns = select_model_features(training)
if (len(training), len(feature_columns), ATTACK_THRESHOLD) != (100_000, 38, 0.5):
    raise ValueError("Training size, feature count, or threshold differs from notebook 05.")

classes = np.array([0, 1])
weights = compute_class_weight("balanced", classes=classes, y=y_train)
model = XGBClassifier(
    n_estimators=100, objective="binary:logistic", tree_method="hist",
    device="cpu", max_depth=6, learning_rate=0.1, random_state=42, n_jobs=2,
)
model.fit(training[feature_columns], y_train, sample_weight=weights[y_train])
if model.get_booster().feature_names != feature_columns:
    raise ValueError("Fitted feature order differs from notebook 05 selection.")
attack_index = list(model.classes_).index(1)

print(f"scikit-learn {sklearn.__version__}, XGBoost {xgboost.__version__}")
print(f"Training sample weights: benign {weights[0]:.6f}, attack {weights[1]:.6f}")
print(f"Locked attack cutoff: >= {ATTACK_THRESHOLD}")
print(f"Ordered features ({len(feature_columns)}): {', '.join(feature_columns)}")

scikit-learn 1.9.1, XGBoost 3.4.1
Training sample weights: benign 21.195422, attack 0.512080
Locked attack cutoff: >= 0.5
Ordered features (38): flow_duration, Header_Length, Protocol Type, Duration, Rate, Drate, fin_flag_number, syn_flag_number, rst_flag_number, psh_flag_number, ack_flag_number, ack_count, syn_count, fin_count, urg_count, rst_count, HTTP, HTTPS, DNS, SSH, TCP, UDP, ARP, ICMP, IPv, Tot sum, Min, Max, AVG, Std, Tot size, IAT, Number, Magnitue, Radius, Covariance, Variance, Weight


**Interpretation.** The fit uses the same training inputs and settings as the validation comparison. The following sections use this fitted model and cutoff for both test scopes.

## Score the full test sample

Generate attack probabilities once for the 25,000 test rows. The confusion matrix uses actual labels down the rows and predicted labels across the columns. Support is the number of actual rows in each class. The benign false-positive rate is the share of benign rows predicted as attack.

In [2]:
def test_metrics(actual, predicted):
    mask = np.ones(len(actual), dtype=bool)
    result = score_predictions(actual, {"XGBoost": predicted}, {"Test": mask}).iloc[0]
    if sum(result[column] for column in ("TN", "FP", "FN", "TP")) != len(actual):
        raise ValueError("Confusion counts do not sum to the test scope size.")
    return result

def show_results(result):
    matrix = pd.DataFrame(
        [[result["TN"], result["FP"]], [result["FN"], result["TP"]]],
        index=pd.Index(["Benign (0)", "Attack (1)"], name="Actual class"),
        columns=pd.Index(["Benign (0)", "Attack (1)"], name="Predicted class"),
        dtype=int,
    )
    display(matrix)
    by_class = pd.DataFrame(
        [
            {
                **{metric: f"{result[f'{label} {metric}']:.4%}" for metric in ("precision", "recall", "F1")},
                "support": int(support),
            }
            for label, support in [
                ("Benign", result["TN"] + result["FP"]),
                ("Attack", result["FN"] + result["TP"]),
            ]
        ],
        index=pd.Index(["Benign (0)", "Attack (1)"], name="Class"),
    )
    display(by_class)
    print(f"Balanced accuracy: {result['Balanced accuracy']:.4%}")
    print(f"Benign false-positive rate: {result['Benign false-positive rate']:.4%}")

test_path = SUBSET_DIR / "ciciot2023_test_stratified_25000_seed42.csv"
test, test_labels = load_features(test_path)
if len(test) != 25_000 or not test.columns.equals(training.columns):
    raise ValueError("Test size or original feature schema differs from the expected subset.")
y_test = binary_targets(test_labels, "Test").to_numpy()
attack_scores = model.predict_proba(test[feature_columns])[:, attack_index]
predictions = (attack_scores >= ATTACK_THRESHOLD).astype(int)

full = test_metrics(y_test, predictions)
show_results(full)
display(Markdown(
    f"**Interpretation.** The detector wrongly flagged **{int(full['FP']):,} benign rows** "
    f"and missed **{int(full['FN']):,} attacks** in the full test sample."
))

Predicted class,Benign (0),Attack (1)
Actual class,,
Benign (0),579,10
Attack (1),116,24295


,precision,recall,F1,support
Class,,,,
Benign (0),83.3094%,98.3022%,90.1869%,589
Attack (1),99.9589%,99.5248%,99.7414%,24411


Balanced accuracy: 98.9135%
Benign false-positive rate: 1.6978%


**Interpretation.** The detector wrongly flagged **10 benign rows** and missed **116 attacks** in the full test sample.

## Exclude exact training matches

Compare parsed numeric tuples across all 46 original features, with labels excluded and without rounding. Count distinct shared patterns and affected test rows separately. Apply the mask to the existing predictions and subtract the full-sample metrics from the filtered metrics using unrounded values.

In [3]:
shared_patterns, matches = validation_overlap(training, test)
matched_benign, matched_attack = np.bincount(y_test[matches], minlength=2)
keep = ~matches
print(f"Distinct shared feature patterns: {shared_patterns:,}")
print(f"Matching test rows: {matches.sum():,} ({matched_benign:,} benign, {matched_attack:,} attack)")
print(f"Nonmatching test rows: {keep.sum():,}")

filtered = test_metrics(y_test[keep], predictions[keep])
show_results(filtered)
delta = filtered - full
count_columns = ["Rows", "TN", "FP", "FN", "TP"]
rate_columns = [
    f"{label} {metric}" for label in ("Benign", "Attack")
    for metric in ("precision", "recall", "F1")
] + ["Balanced accuracy", "Benign false-positive rate"]
changes = {column: f"{int(delta[column]):+d}" for column in count_columns}
changes.update({"Benign support": f"{-matched_benign:+d}", "Attack support": f"{-matched_attack:+d}"})
changes.update({column: f"{100 * delta[column]:+.4f} pp" for column in rate_columns})
display(pd.Series(changes, name="Nonmatching minus full test").to_frame())

display(Markdown(
    f"**Interpretation.** After excluding the matches, the same predictions produced "
    f"**{int(filtered['FP']):,} benign false alarms** and **{int(filtered['FN']):,} missed attacks**. "
    f"Balanced accuracy changed by {changes['Balanced accuracy']} and the benign false-positive "
    f"rate changed by {changes['Benign false-positive rate']}. These results apply to the sampled "
    f"test set. Capture-level independence was not verified."
))

Distinct shared feature patterns: 19
Matching test rows: 19 (1 benign, 18 attack)
Nonmatching test rows: 24,981


Predicted class,Benign (0),Attack (1)
Actual class,,
Benign (0),578,10
Attack (1),116,24277


,precision,recall,F1,support
Class,,,,
Benign (0),83.2853%,98.2993%,90.1716%,588
Attack (1),99.9588%,99.5245%,99.7412%,24393


Balanced accuracy: 98.9119%
Benign false-positive rate: 1.7007%


,Nonmatching minus full test
Rows,-19
TN,-1
FP,+0
FN,+0
TP,-18
Benign support,-1
Attack support,-18
Benign precision,-0.0240 pp
Benign recall,-0.0029 pp
Benign F1,-0.0153 pp


**Interpretation.** After excluding the matches, the same predictions produced **10 benign false alarms** and **116 missed attacks**. Balanced accuracy changed by -0.0016 pp and the benign false-positive rate changed by +0.0029 pp. These results apply to the sampled test set. Capture-level independence was not verified.

## Export the evaluated model and check local predictions

Export the fitted model from this run and record its feature order, class mapping, cutoff, and XGBoost version. Select 16 test rows from each actual class with seed 42, preserve their original row order, and write a feature-only CSV. Run the batch command in a fresh process and compare its output with the notebook's existing predictions for those rows. Require exact class agreement and an absolute score difference of at most `1e-7`.

In [4]:
import json
import subprocess
from io import StringIO

model_dir = SUBSET_DIR.parent / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "xgboost_final.ubj"
metadata_path = model_dir / "xgboost_final_metadata.json"
sample_path = model_dir / "xgboost_final_check_input.csv"
expected_path = model_dir / "xgboost_final_check_expected.csv"

model.save_model(model_path)
metadata = {
    "feature_columns": feature_columns,
    "class_mapping": {"0": "benign", "1": "attack"},
    "attack_threshold": ATTACK_THRESHOLD,
    "xgboost_version": xgboost.__version__,
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

rng = np.random.default_rng(42)
sample_rows = np.sort(np.concatenate([
    rng.choice(np.flatnonzero(y_test == label), size=16, replace=False)
    for label in (0, 1)
]))
# Reverse input columns to exercise the command's saved feature ordering.
test.iloc[sample_rows][feature_columns[::-1]].to_csv(sample_path, index=False)
expected = pd.DataFrame({
    "attack_score": attack_scores[sample_rows].astype(np.float64),
    "predicted_class": predictions[sample_rows],
})
expected.to_csv(expected_path, index=False)

command = [sys.executable, "-m", "src.batch.predict", str(sample_path.relative_to(repo_root))]
result = subprocess.run(command, cwd=repo_root, capture_output=True, text=True)
result.check_returncode()
actual = pd.read_csv(StringIO(result.stdout))
np.testing.assert_allclose(actual["attack_score"], expected["attack_score"], rtol=0, atol=1e-7)
np.testing.assert_array_equal(actual["predicted_class"], expected["predicted_class"])
max_difference = (actual["attack_score"] - expected["attack_score"]).abs().max()
display(Markdown(
    f"**Result.** All {len(expected)} predicted classes matched. "
    f"The largest absolute score difference was {max_difference:.3g} (tolerance: `1e-7`)."
))

**Result.** All 32 predicted classes matched. The largest absolute score difference was 0 (tolerance: `1e-7`).